# Notebook 06-UNSW — Cross-Model SHAP Agreement (Krishna Metrics)

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection  
**Author:** Md Anas Biswas, University of Portsmouth  
**Contribution:** C2 — Cross-model SHAP agreement on UNSW-NB15 (replicates NSL-KDD and CIC-IDS2017 analyses)

## What this notebook does

Applies the six metrics from Krishna et al. 2022 (TMLR) to quantify cross-model SHAP agreement across Random Forest, XGBoost, and DNN architectures on UNSW-NB15. The three pairs:

- **RF ↔ XGB** (tree–tree, expected: high agreement)
- **RF ↔ DNN** (tree–neural, expected: low agreement — "disagreement problem")
- **XGB ↔ DNN** (tree–neural, expected: low agreement)

Computed at three levels:
1. **Binary models** (rf_binary_cw vs xgb_binary_cw vs dnn_binary_cw)
2. **5-class models aggregate** (across all classes)
3. **5-class models per-class** (separately for Normal/DoS/Probe/R2L/U2R) — the most informative slice

## The six Krishna metrics

| Metric | What it measures | Range |
|---|---|---|
| **Feature Agreement** | Fraction of top-k features that appear in both explainers' top-k | [0, 1] |
| **Rank Agreement** | Fraction of top-k features that appear in the same rank position in both | [0, 1] |
| **Sign Agreement** | Fraction of top-k features with the same sign of SHAP in both | [0, 1] |
| **Signed Rank Agreement** | Fraction of top-k features with same sign AND same rank | [0, 1] |
| **Rank Correlation** | Spearman ρ between the two explainers' rankings of the union of top-k features | [-1, 1] |
| **Pairwise Rank Agreement** | For any pair of top-k features, do both explainers agree on which is more important? | [0, 1] |

All metrics computed **per sample** (one value per sample from each explainer comparison), then aggregated to mean across samples.

## Methodology

- **Sample basis:** the 2000 SHAP subsample from Notebook 04 (same indices as Notebooks 04 and 05)
- **k = 10** for top-k feature comparison (matches NSL-KDD/CIC-IDS2017 setup)
- **Feature importance per sample:** sum of |SHAP| across classes for multiclass models, |SHAP[:,1]| for binary
- **Sign:** raw SHAP for the predicted class direction

## Runtime

Pure metric computation, no SHAP recomputation. Total runtime ~3–5 minutes.

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, shutil
shutil.copy('/content/drive/MyDrive/XIDS_Research/.gitconfig',       '/root/.gitconfig')
shutil.copy('/content/drive/MyDrive/XIDS_Research/.git-credentials', '/root/.git-credentials')

PROJECT_DIR = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(PROJECT_DIR)

!git config --global credential.helper 'store --file /content/drive/MyDrive/XIDS_Research/.git-credentials'
!git pull origin main

From https://github.com/anasbiswas1/xids-research
 * branch            main       -> FETCH_HEAD
Already up to date.


## 2. Load SHAP arrays and metadata

In [ ]:
import numpy as np
import json
from pathlib import Path

REPO     = Path(PROJECT_DIR)
PROC_DIR = REPO / 'data/processed/unsw_nb15'
SHAP_DIR = REPO / 'shap_values/unsw_nb15'
AGREE_DIR = SHAP_DIR / 'agreement'
AGREE_DIR.mkdir(parents=True, exist_ok=True)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names = json.load(f)

y_test_5class = np.load(PROC_DIR / 'y_test_5class.npy')
shap_idx = np.load(SHAP_DIR / 'shap_subsample_idx.npy')
y_shap_5class = y_test_5class[shap_idx]

FIVE_CLASS_NAMES = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

# Load SHAP arrays — shape (n_samples, n_features, n_classes)
shap_arrays = {}
for name in ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
             'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote']:
    arr = np.load(SHAP_DIR / f'{name}_shap.npy')
    shap_arrays[name] = arr
    print(f'{name:20s}  {arr.shape}')

rf_binary_cw          (2000, 196, 2)
xgb_binary_cw         (2000, 196, 2)
dnn_binary_cw         (2000, 196, 2)
rf_5class_smote       (2000, 196, 5)


## 3. Define the six Krishna metrics

In [ ]:
from scipy.stats import spearmanr

def top_k_indices(importance_vec, k):
    """Return indices of top-k features by absolute importance, ranked."""
    return np.argsort(np.abs(importance_vec))[::-1][:k]

def feature_agreement(top_a, top_b):
    """Fraction of features appearing in both top-k sets."""
    set_a, set_b = set(top_a), set(top_b)
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def rank_agreement(top_a, top_b):
    """Fraction of top-k positions where both explainers list the SAME feature at the SAME rank."""
    k = min(len(top_a), len(top_b))
    if k == 0:
        return 0.0
    return sum(top_a[i] == top_b[i] for i in range(k)) / k

def sign_agreement(top_a, top_b, sv_a, sv_b):
    """Fraction of common top-k features with the same sign of SHAP in both."""
    common = set(top_a) & set(top_b)
    if not common:
        return 0.0
    same_sign = sum(1 for f in common if np.sign(sv_a[f]) == np.sign(sv_b[f]))
    return same_sign / len(common)

def signed_rank_agreement(top_a, top_b, sv_a, sv_b):
    """Fraction of top-k positions where the same feature appears at the same rank AND with the same sign."""
    k = min(len(top_a), len(top_b))
    if k == 0:
        return 0.0
    matches = 0
    for i in range(k):
        if top_a[i] == top_b[i] and np.sign(sv_a[top_a[i]]) == np.sign(sv_b[top_b[i]]):
            matches += 1
    return matches / k

def rank_correlation(top_a, top_b, imp_a, imp_b):
    """Spearman ρ between the two explainers' rankings of the union of top-k features.
    Returns ρ in [-1, 1]; 0 if union has <2 elements."""
    union = list(set(top_a) | set(top_b))
    if len(union) < 2:
        return 0.0
    rho, _ = spearmanr(imp_a[union], imp_b[union])
    if np.isnan(rho):
        return 0.0
    return float(rho)

def pairwise_rank_agreement(top_a, top_b, imp_a, imp_b):
    """For every pair (i, j) in the union of top-k, do both explainers agree on which is more important?"""
    union = list(set(top_a) | set(top_b))
    if len(union) < 2:
        return 0.0
    total = 0
    agree = 0
    for i in range(len(union)):
        for j in range(i + 1, len(union)):
            a, b = union[i], union[j]
            total += 1
            if (imp_a[a] > imp_a[b]) == (imp_b[a] > imp_b[b]):
                agree += 1
    return agree / total if total > 0 else 0.0

print('Six Krishna metrics defined.')

## 4. Aggregator: compute all six metrics for a pair of SHAP arrays

In [ ]:
K = 10

def aggregate_importance(sv, class_filter=None):
    """sv shape: (n_samples, n_features, n_classes).
    If class_filter is None: importance = sum of |SHAP| across classes (per-sample, per-feature).
    If class_filter is an int c: importance = |SHAP[:, :, c]| (the SHAP for that class only).
    Returns: (n_samples, n_features) importance matrix and (n_samples, n_features) signed-SHAP for sign metrics."""
    if class_filter is None:
        importance = np.abs(sv).sum(axis=-1)
        # For sign: use the dominant-class signed SHAP per sample
        dominant_class = np.abs(sv).sum(axis=1).argmax(axis=-1)
        n = sv.shape[0]
        signed = sv[np.arange(n), :, dominant_class]
    else:
        importance = np.abs(sv[:, :, class_filter])
        signed = sv[:, :, class_filter]
    return importance, signed

def compare_models(sv_a, sv_b, class_filter=None, k=K):
    """Compute the six Krishna metrics, averaged across samples.
    sv_a, sv_b: SHAP arrays with shape (n_samples, n_features, n_classes).
    class_filter: None for aggregate, int c for per-class.
    """
    imp_a, sign_a = aggregate_importance(sv_a, class_filter)
    imp_b, sign_b = aggregate_importance(sv_b, class_filter)
    n = imp_a.shape[0]

    fa_vals, ra_vals, sa_vals, sra_vals, rc_vals, pra_vals = [], [], [], [], [], []

    for i in range(n):
        top_a = top_k_indices(imp_a[i], k)
        top_b = top_k_indices(imp_b[i], k)
        fa_vals.append(feature_agreement(top_a, top_b))
        ra_vals.append(rank_agreement(top_a, top_b))
        sa_vals.append(sign_agreement(top_a, top_b, sign_a[i], sign_b[i]))
        sra_vals.append(signed_rank_agreement(top_a, top_b, sign_a[i], sign_b[i]))
        rc_vals.append(rank_correlation(top_a, top_b, imp_a[i], imp_b[i]))
        pra_vals.append(pairwise_rank_agreement(top_a, top_b, imp_a[i], imp_b[i]))

    return {
        'feature_agreement':       float(np.mean(fa_vals)),
        'rank_agreement':          float(np.mean(ra_vals)),
        'sign_agreement':          float(np.mean(sa_vals)),
        'signed_rank_agreement':   float(np.mean(sra_vals)),
        'rank_correlation':        float(np.mean(rc_vals)),
        'pairwise_rank_agreement': float(np.mean(pra_vals)),
        'n_samples':               n,
    }

print('Aggregator ready.')

## 5. Binary-models agreement (3 pairs)

In [ ]:
binary_pairs = [
    ('rf_binary_cw',  'xgb_binary_cw',  'RF ↔ XGB'),
    ('rf_binary_cw',  'dnn_binary_cw',  'RF ↔ DNN'),
    ('xgb_binary_cw', 'dnn_binary_cw',  'XGB ↔ DNN'),
]

binary_results = {}
for a, b, label in binary_pairs:
    print(f'\n{label} ({a} vs {b})')
    r = compare_models(shap_arrays[a], shap_arrays[b], class_filter=None)
    binary_results[label] = r
    for k, v in r.items():
        if k == 'n_samples':
            continue
        print(f'  {k:28s}  {v:+.4f}')

## 6. 5-class aggregate agreement (3 pairs)

In [ ]:
five_pairs = [
    ('rf_5class_smote',  'xgb_5class_smote',  'RF ↔ XGB'),
    ('rf_5class_smote',  'dnn_5class_smote',  'RF ↔ DNN'),
    ('xgb_5class_smote', 'dnn_5class_smote',  'XGB ↔ DNN'),
]

five_aggregate_results = {}
for a, b, label in five_pairs:
    print(f'\n{label} ({a} vs {b}) — aggregate')
    r = compare_models(shap_arrays[a], shap_arrays[b], class_filter=None)
    five_aggregate_results[label] = r
    for k, v in r.items():
        if k == 'n_samples':
            continue
        print(f'  {k:28s}  {v:+.4f}')

## 7. 5-class per-class agreement

The most informative slice — does cross-model agreement vary by attack class?

In [ ]:
five_per_class_results = {}

for a, b, label in five_pairs:
    print(f'\n{label} ({a} vs {b}) — per-class')
    five_per_class_results[label] = {}
    for cid, cname in enumerate(FIVE_CLASS_NAMES):
        # Restrict to samples where true class == cid for the most diagnostic signal
        mask = (y_shap_5class == cid)
        if mask.sum() < 5:
            print(f'  {cname:8s}: (too few samples, n={mask.sum()})')
            five_per_class_results[label][cname] = None
            continue
        sv_a_sub = shap_arrays[a][mask]
        sv_b_sub = shap_arrays[b][mask]
        r = compare_models(sv_a_sub, sv_b_sub, class_filter=cid)
        five_per_class_results[label][cname] = r
        print(f'  {cname:8s}: FA={r["feature_agreement"]:.3f}  RA={r["rank_agreement"]:.3f}  '
              f'Sign={r["sign_agreement"]:.3f}  ρ={r["rank_correlation"]:+.3f}  '
              f'PairRank={r["pairwise_rank_agreement"]:.3f}  (n={r["n_samples"]})')

## 8. Save all results

In [ ]:
all_results = {
    'binary': binary_results,
    'five_class_aggregate': five_aggregate_results,
    'five_class_per_class': five_per_class_results,
    'config': {
        'top_k': K,
        'n_shap_samples': shap_arrays['rf_binary_cw'].shape[0],
    }
}

with open(AGREE_DIR / 'krishna_agreement.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'Saved: {AGREE_DIR / "krishna_agreement.json"}')

## 9. Summary tables

In [ ]:
import pandas as pd

# Binary models table
rows = []
for label, r in binary_results.items():
    rows.append({
        'Pair': label,
        'FA':       round(r['feature_agreement'], 4),
        'RA':       round(r['rank_agreement'], 4),
        'Sign':     round(r['sign_agreement'], 4),
        'SignRank': round(r['signed_rank_agreement'], 4),
        'RankCorr': round(r['rank_correlation'], 4),
        'PairRank': round(r['pairwise_rank_agreement'], 4),
    })
binary_df = pd.DataFrame(rows)
print('=== Binary models — Krishna agreement (k=10) ===')
print(binary_df.to_string(index=False))
binary_df.to_csv(REPO / 'results/tables/unsw_krishna_binary.csv', index=False)

# 5-class aggregate table
rows = []
for label, r in five_aggregate_results.items():
    rows.append({
        'Pair': label,
        'FA':       round(r['feature_agreement'], 4),
        'RA':       round(r['rank_agreement'], 4),
        'Sign':     round(r['sign_agreement'], 4),
        'SignRank': round(r['signed_rank_agreement'], 4),
        'RankCorr': round(r['rank_correlation'], 4),
        'PairRank': round(r['pairwise_rank_agreement'], 4),
    })
five_df = pd.DataFrame(rows)
print('\n=== 5-class models — Krishna agreement aggregate (k=10) ===')
print(five_df.to_string(index=False))
five_df.to_csv(REPO / 'results/tables/unsw_krishna_5class_aggregate.csv', index=False)

# Per-class table — focus on rank correlation and sign agreement (the most diagnostic)
rows = []
for label, by_class in five_per_class_results.items():
    for cname, r in by_class.items():
        if r is None:
            continue
        rows.append({
            'Pair': label,
            'Class': cname,
            'FA':       round(r['feature_agreement'], 4),
            'Sign':     round(r['sign_agreement'], 4),
            'RankCorr': round(r['rank_correlation'], 4),
            'n':        r['n_samples'],
        })
perclass_df = pd.DataFrame(rows)
print('\n=== 5-class models — Krishna agreement per class (k=10) ===')
print(perclass_df.to_string(index=False))
perclass_df.to_csv(REPO / 'results/tables/unsw_krishna_5class_per_class.csv', index=False)

## 10. Visualisation — agreement heatmap

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left: aggregate binary vs 5-class for rank correlation
metrics_to_plot = ['feature_agreement', 'rank_agreement', 'sign_agreement', 'rank_correlation', 'pairwise_rank_agreement']
labels_short    = ['Feature', 'Rank', 'Sign', 'Rank Corr', 'Pair Rank']
pairs = list(binary_results.keys())

data_binary = np.array([[binary_results[p][m] for m in metrics_to_plot] for p in pairs])
data_five   = np.array([[five_aggregate_results[p][m] for m in metrics_to_plot] for p in pairs])

# Plot binary
im0 = axes[0].imshow(data_binary, cmap='RdYlGn', vmin=-0.3, vmax=1.0, aspect='auto')
axes[0].set_xticks(range(len(metrics_to_plot)))
axes[0].set_xticklabels(labels_short, rotation=30, ha='right')
axes[0].set_yticks(range(len(pairs)))
axes[0].set_yticklabels(pairs)
axes[0].set_title('Binary models — Krishna agreement (k=10)')
for i in range(len(pairs)):
    for j in range(len(metrics_to_plot)):
        axes[0].text(j, i, f'{data_binary[i,j]:+.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im0, ax=axes[0], fraction=0.04)

# Plot 5-class aggregate
im1 = axes[1].imshow(data_five, cmap='RdYlGn', vmin=-0.3, vmax=1.0, aspect='auto')
axes[1].set_xticks(range(len(metrics_to_plot)))
axes[1].set_xticklabels(labels_short, rotation=30, ha='right')
axes[1].set_yticks(range(len(pairs)))
axes[1].set_yticklabels(pairs)
axes[1].set_title('5-class models — Krishna agreement aggregate (k=10)')
for i in range(len(pairs)):
    for j in range(len(metrics_to_plot)):
        axes[1].text(j, i, f'{data_five[i,j]:+.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im1, ax=axes[1], fraction=0.04)

plt.suptitle('UNSW-NB15 — Cross-model SHAP agreement (Krishna et al., 2022)', fontsize=13, y=1.02)
plt.tight_layout()
fig_path = REPO / 'results/figures/unsw_krishna_agreement_heatmap.png'
plt.savefig(fig_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# Per-class rank correlation heatmap — the key diagnostic figure for the paper
fig, ax = plt.subplots(figsize=(10, 4))

matrix = np.full((len(five_per_class_results), len(FIVE_CLASS_NAMES)), np.nan)
for i, (label, by_class) in enumerate(five_per_class_results.items()):
    for j, cname in enumerate(FIVE_CLASS_NAMES):
        r = by_class.get(cname)
        if r is not None:
            matrix[i, j] = r['rank_correlation']

im = ax.imshow(matrix, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(FIVE_CLASS_NAMES)))
ax.set_xticklabels(FIVE_CLASS_NAMES)
ax.set_yticks(range(len(five_per_class_results)))
ax.set_yticklabels(list(five_per_class_results.keys()))
ax.set_title('UNSW-NB15 — Per-class Krishna rank correlation (k=10)')
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix[i, j]
        text = 'n/a' if np.isnan(v) else f'{v:+.2f}'
        ax.text(j, i, text, ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()

pc_fig_path = REPO / 'results/figures/unsw_krishna_per_class_rank_corr.png'
plt.savefig(pc_fig_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved: {pc_fig_path}')

## 11. Commit and push

In [ ]:
os.chdir(PROJECT_DIR)
!git add notebooks/06_unsw_shap_agreement.ipynb \
         results/figures/unsw_krishna_agreement_heatmap.png \
         results/figures/unsw_krishna_per_class_rank_corr.png \
         results/tables/unsw_krishna_binary.csv \
         results/tables/unsw_krishna_5class_aggregate.csv \
         results/tables/unsw_krishna_5class_per_class.csv
!git commit -m "Notebook 06-UNSW: Krishna et al. 2022 cross-model SHAP agreement, six metrics, binary+5class+per-class (C2)"
!git push origin main

---

## How to read the results

**Tree-tree agreement (RF ↔ XGB):** Expected to be high — both are tree ensembles trained on the same features, so they tend to surface similar feature importance patterns. Feature Agreement and Pairwise Rank Agreement should be 0.5+. Rank Correlation should be moderately positive (+0.3 to +0.6).

**Tree-DNN agreement (RF ↔ DNN, XGB ↔ DNN):** Expected to be weak — different architectures process features differently. Rank Correlation may be near zero or even negative. Sign Agreement may still be high (both architectures agree on the *direction* of feature effects, even if they disagree on ordering).

**Per-class:** The most diagnostic finding. On NSL-KDD, sign agreement on rare classes (R2L, U2R) dropped from 0.96 → 0.78 — meaning models *agree on what features matter* but *disagree on whether those features push the prediction toward attack or away from it* for rare classes. If UNSW replicates this pattern, the paper has a robust three-dataset finding.

## Cross-dataset comparison

| Pair | NSL-KDD Rank Corr | UNSW-NB15 Rank Corr |
|---|---:|---:|
| RF ↔ XGB  | +0.47 to +0.55 | *fill in from this notebook's output* |
| RF ↔ DNN  | −0.30 to +0.06 | *fill in* |
| XGB ↔ DNN | −0.17 to −0.10 | *fill in* |

If the pattern replicates (positive RF-XGB, near-zero or negative tree-DNN), the cross-dataset finding is solid.

## Next step

**This is the last UNSW notebook.** After running it:
1. Update `cross_dataset_comparison.md` Tables 3, 4, 5 with the new numbers
2. Update `UNSW_NB15_design_decisions.md` if any methodology decisions need refining based on the agreement results
3. Prepare for the June 1 supervisor meeting